# 🛰️ AntarikshDrishti — Live Cloud Deployment
### Smart India Hackathon PS 26227 | 1-Click Free Cloud Runner

This Google Colab notebook launches the full-stack **AntarikshDrishti** satellite intelligence workstation with zero configuration:
- ✅ **React Mission Control UI** (Embedded interactive map, multi-temporal change detection)
- ✅ **OpenCLIP ViT-B/16 Semantic Vector Search**
- ✅ **FastAPI REST API Engine** (`/api/v1`)
- ✅ **32 Multi-Spectral Mumbai Sentinel-2 Scenes**
- ✅ **Global HTTPS URL via Cloudflare Tunnel**

👉 **To launch:** Click **Runtime** > **Run all** (or press `Ctrl+F9`).

In [ ]:
# Step 1: Clone Repository
import os
if not os.path.exists("/content/AntarikshDrishti"):
    !git clone https://github.com/Overfitters-2026/AntarikshDrishti.git /content/AntarikshDrishti
%cd /content/AntarikshDrishti


In [ ]:
# Step 2: Install Dependencies & Cloudflare Tunnel
!pip install -q fastapi uvicorn[standard] rasterio tifffile qdrant-client open_clip_torch scikit-learn scikit-image joblib aiosqlite pydantic-settings portalocker
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
print("Dependencies and tunnel installed successfully!")


In [ ]:
# Step 3: Launch Full-Stack Platform & Expose Public HTTPS URL
import subprocess
import time
import re
import os

# Ensure working directory is Geo_Search
os.chdir("/content/AntarikshDrishti/Geo_Search")

# Terminate any existing background processes
!pkill -f uvicorn > /dev/null 2>&1
!pkill -f cloudflared > /dev/null 2>&1

# 1. Start FastAPI backend (which also serves prebuilt React UI at root)
backend_proc = subprocess.Popen(
    ["uvicorn", "main:app", "--host", "127.0.0.1", "--port", "8000"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
print("Starting backend server...")
time.sleep(4)

# 2. Start Cloudflare Tunnel to expose port 8000
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

public_url = None
start_time = time.time()
print("Generating secure public HTTPS link...")
while time.time() - start_time < 35:
    line = tunnel_proc.stderr.readline()
    if "trycloudflare.com" in line:
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match:
            public_url = match.group(0)
            break

if public_url:
    print("=" * 65)
    print(f"🎉 ANTARIKSHDRISHTI IS LIVE ONLINE:")
    print(f"👉 Interactive UI:  {public_url}/")
    print(f"👉 REST API Docs:   {public_url}/docs")
    print("=" * 65)
else:
    print("Tunnel initialization timed out. Please rerun this cell.")

# Keep cell active
try:
    backend_proc.wait()
except KeyboardInterrupt:
    backend_proc.kill()
    tunnel_proc.kill()
    print("Server stopped.")
